# Pipeline development – end-to-end and testable

**How to use:**
- Run **Config** and **Load models** once per session.
- Then either run **"Run full pipeline"** (one cell) or run **Steps 1–7** in order for step-by-step execution.
- To test a specific part: change parameters and re-run from that step (e.g. re-run from Direction change or Speed); no need to re-load models or re-run heavy detection steps.
- Set `TASK_ID = None` for no DB writes; set a valid task id when you want to persist to the database.

In [ ]:
# Config – run once, then run "Load models" and either full pipeline or steps 1–7
import os

VIDEO_PATH = "path/to/your/video.mp4"  # required
DEVICE = "cuda"  # or "cpu"
TASK_ID = None   # None = no DB; set to an int to write to DB
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Imports
import time
import cv2
import numpy as np
from collections import defaultdict
from scipy.spatial.distance import euclidean

from src.core.ball_detector import BallDetector
from src.core.bounce_detector import BounceDetector
from src.core.court_detection_net import CourtDetectorNet
from src.core.person_detector import PersonDetector
from src.core.process_video import (
    process_video,
    get_detections_from_video,
    get_valid_scenes,
    get_sources_from_source_indices,
    get_shot_type,
    cleanup_memory,
)
from src.core.get_direction_change_indices import get_direction_change_indices
from src.core.utils import (
    get_court_img,
    generate_player_heatmap,
    perspective_transform_point,
    scene_detect,
)
from src.schemas.speed_at import SpeedAt
from src.db.utils import (
    save_ball_track_in_db,
    save_bounces_in_db,
    save_direction_change_indices_in_db,
    save_player_positions_in_db,
    save_speed_in_db,
    save_thumbnail_in_db,
    save_video_paths_in_db,
)

In [ ]:
# Load models – run once per session
ball_detector = BallDetector("./src/track_net_weights.pt", DEVICE)
court_detector = CourtDetectorNet("./src/model_tennis_court_det.pt", DEVICE)
person_detector = PersonDetector(DEVICE)
bounce_detector = BounceDetector("./src/ctb_regr_bounce.cbm")
print(f"Models loaded on {DEVICE}.")

## Step 1: Scene detection

In [ ]:
scenes = scene_detect(VIDEO_PATH)
print("Scenes:", scenes)
max_diff = max(scenes, key=lambda x: x[1] - x[0])
thumbnail_index = max_diff[0]

## Step 2: Valid scenes (court filter)

In [ ]:
valid_scenes = get_valid_scenes(court_detector, VIDEO_PATH, scenes)
print(f"Valid scenes: {len(valid_scenes)} / {len(scenes)}")

## Step 3: Detections (ball, court, person, bounces)

In [ ]:
ball_track, bounces, homography_matrices, kps_court, player_top, player_bottom = get_detections_from_video(
    ball_detector, court_detector, person_detector, bounce_detector,
    task_id=TASK_ID, video_path=VIDEO_PATH, valid_scenes=valid_scenes,
)
print(f"Ball track: {len(ball_track)}, bounces: {len(bounces)}")
device = getattr(ball_detector, "device", "cpu")
cleanup_memory(device)

## Step 4: Transform track and serve marking

In [ ]:
transformed_track = [
    perspective_transform_point(point, homography_matrices[i])
    for i, point in enumerate(ball_track)
]
serve_frames = set()
scene_starts = {s[0] for s in valid_scenes}
for bounce_index in sorted(bounces):
    for scene_start, scene_end in valid_scenes:
        if scene_start <= bounce_index < scene_end:
            if scene_start in scene_starts:
                serve_frames.add(bounce_index)
                scene_starts.discard(scene_start)
            break
print(f"Serves: {len(serve_frames)}")
if TASK_ID is not None:
    save_ball_track_in_db(TASK_ID, transformed_track)
    save_bounces_in_db(TASK_ID, {index: transformed_track[index] for index in bounces}, serve_frames)
    save_player_positions_in_db(TASK_ID, player_top, player_bottom)
cleanup_memory(device)

## Step 5: Direction change indices

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

direction_change_indices = list(get_direction_change_indices(ball_track, buffer_length=8))
indices = []
for i, ind in enumerate(sorted(direction_change_indices)):
    if i == 0:
        indices.append(ind)
        continue
    if ind - indices[-1] < 6:
        pass
    else:
        indices.append(ind)
change_before_bounce = defaultdict(list)
outer = 0
for i in bounces:
    for j in indices[outer:]:
        if j < i:
            frame_diff = i - j
            if frame_diff >= 15 and frame_diff <= int(2 * fps):
                change_before_bounce[i].append((j, transformed_track[j]))
    outer += 1
direction_change_indices = indices
if TASK_ID is not None:
    save_direction_change_indices_in_db(TASK_ID, {index: ball_track[index] for index in direction_change_indices})
cleanup_memory(device)

## Step 6: Speed calculation

In [ ]:
PIXEL_TO_METER_RATIO = 1 / 101.5
speed_before_bounce = {}
for bounce_index, source_indices in change_before_bounce.items():
    destination = transformed_track[bounce_index]
    if destination[0] is None:
        continue
    sources, inds = get_sources_from_source_indices(transformed_track, source_indices)
    shot_type = get_shot_type(
        sources, destination,
        [player_top[k] for k in inds],
        [player_bottom[k] for k in inds],
    )
    pixel_distance = np.mean([euclidean(source, destination) for source in sources])
    meter_distance = pixel_distance * PIXEL_TO_METER_RATIO
    time_difference = (bounce_index - max(source_indices, key=lambda x: x[0])[0]) / float(fps)
    speed_before_bounce[bounce_index] = SpeedAt(
        speed=(meter_distance / time_difference) * 3.6,
        time_diff=time_difference,
        timestamp=bounce_index / float(fps),
        distance=meter_distance,
        shot_type=shot_type,
    )
speed_indices = sorted(speed_before_bounce.keys(), reverse=True)
if TASK_ID is not None:
    save_speed_in_db(TASK_ID, speed_before_bounce)
cleanup_memory(device)

## Step 7: Annotated video, minimap, heatmaps

In [ ]:
task_suffix = TASK_ID if TASK_ID is not None else "dev"
thumbnail_path = os.path.join(OUTPUT_DIR, f"output_{task_suffix}_thumbnail_{time.time()}.jpg")
output_path = os.path.join(OUTPUT_DIR, f"output_{task_suffix}_{time.time()}.mp4")
minimap_path = os.path.join(OUTPUT_DIR, f"output_{task_suffix}_minimap_{time.time()}.mp4")

input_cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(input_cap.get(cv2.CAP_PROP_FRAME_COUNT))
valid_frame_set = set()
for start, end in valid_scenes:
    valid_frame_set.update(range(start, end))

minimap = get_court_img()
width_minimap, height_minimap = 166, 350
out_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (1280, 720))
speed_indices_copy = list(speed_indices)

for i in range(total_frames):
    ret, frame = input_cap.read()
    if not ret:
        break
    frame = cv2.resize(frame, (1280, 720))
    if i > 0 and i % 1000 == 0:
        cleanup_memory(device)
    if i == thumbnail_index:
        cv2.imwrite(thumbnail_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
        if TASK_ID is not None:
            save_thumbnail_in_db(TASK_ID, thumbnail_path)
    if i not in valid_frame_set:
        out_writer.write(frame)
        continue
    if ball_track[i][0] is not None:
        r, color = (10, (0, 0, 255)) if i in direction_change_indices else (5, (0, 255, 0))
        frame = cv2.circle(frame, (int(ball_track[i][0]), int(ball_track[i][1])), r, color, 2)
    minimap_frame = minimap.copy()
    if ball_track[i][0] is not None and homography_matrices[i] is not None:
        pt = transformed_track[i]
        minimap_frame = cv2.circle(minimap_frame, (int(pt[0]), int(pt[1])), 0, (0, 255, 0), 30)
    if i in bounces and homography_matrices[i] is not None and ball_track[i][0] is not None:
        pt = transformed_track[i]
        minimap_frame = cv2.circle(minimap_frame, (int(pt[0]), int(pt[1])), 0, (0, 255, 255), 50)
        minimap = cv2.circle(minimap, (int(pt[0]), int(pt[1])), 0, (0, 255, 255), 50)
    inv_mat = homography_matrices[i]
    if inv_mat is not None:
        for player, color in [(player_top[i], (0, 0, 255)), (player_bottom[i], (255, 0, 0))]:
            if player is not None:
                foot = np.array(player[1], dtype=np.float32).reshape(1, 1, 2)
                court_pt = cv2.perspectiveTransform(foot, inv_mat)
                minimap_frame = cv2.circle(minimap_frame, (int(court_pt[0, 0, 0]), int(court_pt[0, 0, 1])), 0, color, 60)
    minimap_resized = cv2.resize(minimap_frame, (width_minimap, height_minimap))
    h, w = frame.shape[:2]
    frame[30 : 30 + height_minimap, (w - 30 - width_minimap) : (w - 30), :] = minimap_resized
    if speed_indices_copy:
        si = speed_indices_copy[-1]
        s = speed_before_bounce[si]
        frame = cv2.putText(frame, f"Speed: {s.speed:.2f} km/hr Time: {s.time_diff:.2f} s Distance: {s.distance:.2f} m Shot: {s.shot_type}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        if i > speed_indices_copy[-1] and len(speed_indices_copy) > 1:
            speed_indices_copy.pop()
    out_writer.write(frame)
out_writer.release()
input_cap.release()
print("Annotated video saved:", output_path)

In [ ]:
# Minimap-only video
minimap = get_court_img()
h, w, _ = minimap.shape
minimap_out = cv2.VideoWriter(minimap_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
for i in range(len(transformed_track) - 1):
    minimap_copy = minimap.copy()
    if transformed_track[i][0] is not None and transformed_track[i + 1][0] is not None:
        color = (0, 0, 255) if i in direction_change_indices else (0, 255, 0)
        minimap_copy = cv2.circle(minimap_copy, (int(transformed_track[i][0]), int(transformed_track[i][1])), 0, color, 10)
        minimap_copy = cv2.line(minimap_copy, (int(transformed_track[i][0]), int(transformed_track[i][1])), (int(transformed_track[i + 1][0]), int(transformed_track[i + 1][1])), color, 2)
    minimap_out.write(minimap_copy)
    if i > 0 and i % 1000 == 0:
        cleanup_memory(device)
minimap_out.release()
print("Minimap video saved:", minimap_path)

# Heatmaps
top_court_points, bottom_court_points = [], []
for i in range(len(player_top)):
    inv_mat = homography_matrices[i]
    if inv_mat is None:
        continue
    for player_data, points_list in [(player_top[i], top_court_points), (player_bottom[i], bottom_court_points)]:
        if player_data is not None:
            try:
                foot = np.array(player_data[1], dtype=np.float32).reshape(1, 1, 2)
                cp = cv2.perspectiveTransform(foot, inv_mat)
                points_list.append((float(cp[0, 0, 0]), float(cp[0, 0, 1])))
            except cv2.error:
                continue
heatmap_top_path = os.path.join(OUTPUT_DIR, f"output_{task_suffix}_{time.time()}_heatmap_top.png")
heatmap_bottom_path = os.path.join(OUTPUT_DIR, f"output_{task_suffix}_{time.time()}_heatmap_bottom.png")
cv2.imwrite(heatmap_top_path, generate_player_heatmap(top_court_points))
cv2.imwrite(heatmap_bottom_path, generate_player_heatmap(bottom_court_points))
print(f"Heatmaps saved: {heatmap_top_path}, {heatmap_bottom_path}")

if TASK_ID is not None:
    save_video_paths_in_db(TASK_ID, "dev", output_path, minimap_path)

## Run full pipeline (one shot)

Runs the entire pipeline in one go; use when you don't need to inspect or re-run individual steps.

In [ ]:
process_video(
    ball_detector, court_detector, person_detector, bounce_detector,
    video_path=VIDEO_PATH, task_id=TASK_ID, name="dev",
)